In [1]:
import numpy as np
import xarray as xr
import os
from matplotlib import pyplot as plt
from tqdm import tqdm
from wavediffusion.waveana import assemble 
import cartopy.crs as ccrs
from wavediffusion.waveana import add_lat_lon

%load_ext autoreload
%autoreload 2

# Open a random raw file to get lat and lon
ds = xr.open_dataset('/global/homes/j/jiarongw/scratch_folder/wave_data/raw/LOPS_WW3-GLOB-30M_202109.nc')
lat_deg = ds.latitude[1:-2].values
lon_deg = ds.longitude.values

In [2]:
############ Prepare mask and lat weights ############
# Weights computed and saved with wavedata.ipynb
wlat = np.load('/global/homes/j/jiarongw/scratch_folder/wave_data/wlat.npy')
# weighted and masked RMSE
def weighted_mse(a, b, mask, w):
    diff2 = (a - b) ** 2
    return np.sum(diff2[mask] * w[mask]) / np.sum(w[mask])
# weighted and masked spread / skill
def weighted_meanvar(std, mask, w):
    var = std ** 2
    return np.sum(var[mask] * w[mask]) / np.sum(w[mask])    

In [9]:
# Model
OPTION = 3
# case = '_nohist'
case = ''
epoch = 6

# Time 
year = 2004
month = 4

path = f'/global/homes/j/jiarongw/scratch_folder/final/temp/OPTION{OPTION}{case}_sigma100_epoch{epoch}_{year}{month:02d}/'
print(f'Reading from {path} snapshot {year}-{month:02d}...')

if OPTION == 1:
    var_names = ['hs', 'tp', 'thetap']
elif OPTION == 2:
    var_names = ['hs', 'uss', 'vss', 'mssd', 'mssc']
elif OPTION == 3:
    var_names = ['hs_p1', 'tp_p1', 'thetap_p1', 'hs_p2', 'tp_p2', 'thetap_p2', 'ci']

Reading from /global/homes/j/jiarongw/scratch_folder/final/temp/OPTION3_sigma100_epoch6_200404/ snapshot 2004-04...


In [8]:
mse_vars = {name: [] for name in var_names}
mse_sample_vars = {name: [] for name in var_names}
std_vars = {name: [] for name in var_names}

for index in tqdm(range(0, 200, 8)):   
    x_truth = np.load(f'{path}truth_{index}.npy')
    x_sample = np.load(f'{path}sample_{index}.npy')
    x_mean = np.load(f'{path}mean_{index}.npy')
    std = np.load(f'{path}std_{index}.npy')
    icymask = np.load(f'{path}icymask_{index}.npy')

    # Use the ice mask?
    for name in var_names:
         mse_vars[name].append(weighted_mse(x_truth[var_names.index(name)], x_mean[var_names.index(name)], icymask, wlat))
         mse_sample_vars[name].append(weighted_mse(x_truth[var_names.index(name)], x_sample[var_names.index(name)], icymask, wlat))        
         std_vars[name].append(weighted_meanvar(std[var_names.index(name)], icymask, wlat))

for name in var_names:
    mse_vars[name] = np.array(mse_vars[name])
    mse_sample_vars[name] = np.array(mse_sample_vars[name])
    std_vars[name] = np.array(std_vars[name])
    print(f'{name} mean mse: {mse_vars[name].mean()**0.5:.4f} \\pm {np.std(mse_vars[name]**0.5):.4f}')
    print(f'{name} sample mse: {mse_sample_vars[name].mean()**0.5:.4f} \\pm {np.std(mse_sample_vars[name]**0.5):.4f}')
    print(f'{name} ssr: {(std_vars[name].mean() / mse_vars[name].mean())**0.5:.4f}')    

100%|██████████| 25/25 [00:03<00:00,  8.04it/s]

hs_p1 mean mse: 0.4736 \pm 0.0442
hs_p1 sample mse: 0.5677 \pm 0.0524
hs_p1 ssr: 0.6953
tp_p1 mean mse: 2.1027 \pm 0.1012
tp_p1 sample mse: 2.6986 \pm 0.2041
tp_p1 ssr: 0.7892
thetap_p1 mean mse: 52.1464 \pm 3.2708
thetap_p1 sample mse: 66.7196 \pm 4.1057
thetap_p1 ssr: 0.8045
hs_p2 mean mse: 0.3906 \pm 0.0285
hs_p2 sample mse: 0.4899 \pm 0.0307
hs_p2 ssr: 0.7431
tp_p2 mean mse: 3.4358 \pm 0.1991
tp_p2 sample mse: 4.1932 \pm 0.2359
tp_p2 ssr: 0.7167
thetap_p2 mean mse: 80.9530 \pm 4.4993
thetap_p2 sample mse: 106.2592 \pm 4.4411
thetap_p2 ssr: 0.8587
ci mean mse: 0.3496 \pm 0.0172
ci sample mse: 0.4625 \pm 0.0203
ci ssr: 0.8622
